## Imports

In [0]:
from pyspark.sql.functions import col, current_timestamp, monotonically_increasing_id
from botocore.exceptions import NoCredentialsError
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from datetime import datetime
import pandas as pd
import boto3

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()

## Get the needed Parameters and Tables

In [0]:
Ref_Calendar = spark.sql(f"""                    
                        select distinct
                              CalendarDate
                            , PreviousBusinessDay
                            , FirstDateOfMonth
                            , CurrentMonthLastWorkingDate
                            , PreviousMonthLastWorkingDate
                        from `{env}-cdh`.reference.ref_calendar
                    """)

Ref_Calendar.createOrReplaceTempView("Ref_Calendar")

In [0]:
Businessdate = spark.sql(f"""                    
                        select PreviousBusinessDay
                        from Ref_Calendar
                        where CalendarDate = cast(getdate() as date)
                    """).first()[0]

In [0]:
run_date = dbutils.widgets.get("run_date")

Calendar = spark.sql(f"""                    
                        select distinct
                              CalendarDate
                            , FirstDateOfMonth
                            , CurrentMonthLastWorkingDate AS LastDateOfMonth
                            , PreviousMonthLastWorkingDate
                        from Ref_Calendar
                        where CalendarDate=(
                                           select
                                                case 
                                                    when try_cast('{run_date}' as date) is null then PreviousMonthLastWorkingDate
                                                    else CurrentMonthLastWorkingDate
                                                end BaseDate 
                                            from Ref_Calendar
                                            where CalendarDate = coalesce(try_cast('{run_date}' as date),cast(getdate() as date))
                                            )
                    """)

Calendar.createOrReplaceTempView("Calendar")

In [0]:
prvDate = spark.sql(f""" 
                        select PreviousMonthLastWorkingDate
                        from Calendar
                    """).first()[0]

In [0]:
curDate = spark.sql(f""" 
                        select LastDateOfMonth
                        from Calendar
                    """).first()[0]

In [0]:
matchdt = spark.sql(f"""                    
                        select cast(c.CalendarDate as date)
                        from `{env}-cdh`.reference.ref_calendar c
                        cross join Calendar dt 
						where BusinessDayOfMonth = 2
                        and year(c.CalendarDate) = year(add_months(dt.FirstDateOfMonth, 1))
                        and month(c.CalendarDate) = month(add_months(dt.FirstDateOfMonth, 1))
                    """).first()[0]

In [0]:
Portfolio = spark.sql(f"""                    
                        select distinct
                              PortfolioId
                            , PortfolioName
                            , ProfitCenter
                        from `{env}-cdh`.primary.portfolio
                        where BusinessDate = '{Businessdate}'
                        and profitCenter is not null
                        and PortfolioId not in (31,32) -- exclude core Wholesale/Retail
                        and (Portfolioid < 600  or PortfolioId = 605) --exclude reoco except cosmos
                        and Portfolioid != 10 --exclude N/A
                        and Portfolioid not in (18,27,36,49) --common customer and closed portfolios
                    """)

Portfolio.createOrReplaceTempView("Portfolio")

In [0]:
DExposure = spark.sql(f"""                    
                        select distinct
                               ExposureKey
                            ,  PortfolioName
                            , BusinessDate
                            , Currency
                            , GBV
                        from `{env}-cdh`.derivative.DExposure
                        where IsActiveFlag = True
                    """)

DExposure.createOrReplaceTempView("DExposure")

In [0]:
DCollection = spark.sql(f"""                    
                        select distinct
                              PortfolioName
                            , DCollectionKey
                            , ActualDate
                            , Currency
                            , Amount
                            , BusinessDate
                        from `{env}-cdh`.derivative.DCollection c
                        where c.BusinessDate = '{Businessdate}'
                        and c.FinalReason <> 'Not Matched'
                        and coalesce(MatchedDate,'1900-01-01') <= '{matchdt}'
                    """)

DCollection.createOrReplaceTempView("DCollection")

## Calculate AuM - GBV (€bn)

In [0]:
GBV = spark.sql(f"""                    
                    select PortfolioId
                         , SUM(`{env}-cdh`.derivative.converttoeur(dex.Currency, dex.GBV, dex.BusinessDate)) AS GBVEuro
                    from DExposure dex
                    inner join Portfolio p
                    on p.PortfolioName = dex.PortfolioName
                    where dex.BusinessDate = '{prvDate}'
                    group by all
                    union
                    select PortfolioId
                         , SUM(`{env}-cdh`.derivative.converttoeur(dex.Currency, dex.GBV, dex.BusinessDate)) as GBVEuro
                    from DExposure dex
                    inner join Portfolio p
                    on p.PortfolioName = dex.PortfolioName
                    where dex.BusinessDate = '{curDate}'
                    group by all
                """)

GBV.createOrReplaceTempView("GBV")

In [0]:
AuM_GBV_Export = spark.sql(f"""   
                        select year(LastDateOfMonth)          AS Year
                             , month(LastDateOfMonth)         AS Period
                             , 'Cepal Serv'                   AS Entity
                             , 'AuM - GBV (€bn)'              AS Account
                             , cast(profitCenter as string)   AS Profits_Center
                             , cast(null as int)              AS BatchNumber
                             , cast(avg_gbv as decimal(19,2)) AS Amount 
                        from (
                            select ProfitCenter, sum(avg_gbv) avg_gbv
                            from (        
                                select PortfolioId, GBV/cnt AS avg_gbv
                                from (
                                    select PortfolioId, count(1) cnt, sum(GBVEuro) GBV 
                                    from GBV g
                                    where GBVEuro is not null
                                    group by all
                                ) AS d
                                where d.cnt > 0
                            ) AS g
                            inner join Portfolio p
                            on p.PortfolioId = g.PortfolioId
                            group by all
                        ) AS A
                        cross join Calendar
                    """)

AuM_GBV_Export.createOrReplaceTempView("AuM_GBV_Export")

## Calculate Collections (€mn)

In [0]:
CalcCollection = spark.sql(f"""   
                            select distinct 
                                  p.PortfolioId
                                , SUM(`{env}-cdh`.derivative.converttoeur(c.Currency, c.Amount, c.BusinessDate)) AS collectionAmnEuro
                            from DCollection c
                            inner join Portfolio p
                            on p.PortfolioName = c.PortfolioName
                            cross join Calendar dt
                            where month(c.ActualDate)=month(dt.lastDateofmonth)
                            and year(c.ActualDate)=year(dt.lastDateofmonth)
                            group by all
                        """)
                        
CalcCollection.createOrReplaceTempView("CalcCollection")

In [0]:
Collections_Export = spark.sql(f"""  
                                select distinct
                                       year(LastDateOfMonth)                    AS Year
                                     , month(LastDateOfMonth)                   AS Period
                                     , 'Cepal Serv'                             AS Entity
                                     , 'Collections (€mn)'                      AS Account
                                     , cast(profitCenter as string)             AS Profits_Center
                                     , cast(null as int)                        AS BatchNumber
                                     , cast(collectionAmnEuro as decimal(19,2)) AS Amount 
                                from (
                                    select profitCenter, sum(collectionAmnEuro) collectionAmnEuro
                                    from (
                                        select  p.ProfitCenter, coalesce(collectionAmnEuro, 0) AS collectionAmnEuro
                                        from Portfolio p
                                        left join CalcCollection c
                                        on p.PortfolioId = c.PortfolioId
                                    ) as d 
                                    group by profitCenter
                                ) as e
                                cross join Calendar
                            """)

## Create the Final Output

In [0]:
PnRToolExport = AuM_GBV_Export.union(Collections_Export).distinct().orderBy('Account','Profits_Center')
PnRToolExport = PnRToolExport.withColumn('RowID', monotonically_increasing_id() + 1)
PnRToolExport = PnRToolExport.withColumn('InsertDate', current_timestamp())

# Reorder columns
columns_order = ['RowID', 'Year', 'Period', 'Entity', 'Account', 'Profits_Center', 'InsertDate', 'BatchNumber', 'Amount']
PnRToolExport = PnRToolExport.select(columns_order)

In [0]:
Output_path = f"`{env}-cdh-delivery`.it.PnRToolExport"
PnRToolExport.write.mode("overwrite").saveAsTable(Output_path)